# Notebook (LLM ON Kaggel (GPU))

## Setup (Libraries)

In [1]:
!pip install -q flask pyngrok transformers accelerate bitsandbytes

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.1/43.1 MB 42.1 MB/s eta 0:00:00:00:0100:01


## Import Librries

In [2]:
import torch
import threading
import time
import traceback
import socket
import requests
from flask import Flask, request, jsonify
from pyngrok import ngrok
from transformers import AutoModelForCausalLM, AutoTokenizer

## LLM Model

In [3]:
MODEL_NAME = "mistralai/Mistral-7B-Instruct-v0.3"

print("Loading tokenizer...")
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
print("Tokenizer loaded successfully ✅")

print("Loading model (this may take a few minutes)...")
model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    torch_dtype=torch.float16,
    device_map="auto"
)
print("Model loaded successfully ✅")

app = Flask(__name__)


@app.route("/generate", methods=["POST"])
def generate():
    try:
        data = request.get_json()
        prompt = data.get("prompt", "")
        max_new_tokens = data.get("max_new_tokens", 512)
        temperature = data.get("temperature", 0.3)

        messages = [{"role": "user", "content": prompt}]

        inputs = tokenizer.apply_chat_template(
            messages,
            return_tensors="pt",
            add_generation_prompt=True,
            return_dict=True
        ).to(model.device)

        with torch.no_grad():
            output_ids = model.generate(
                **inputs,
                max_new_tokens=max_new_tokens,
                temperature=temperature,
                do_sample=True,
                pad_token_id=tokenizer.eos_token_id
            )

        generated = output_ids[0][inputs["input_ids"].shape[-1]:]
        text = tokenizer.decode(generated, skip_special_tokens=True)

        return jsonify({"response": text})

    except Exception as e:
        tb = traceback.format_exc()
        print(tb)
        return jsonify({"error": str(e), "traceback": tb}), 500

Loading tokenizer...


config.json:   0%|          | 0.00/601 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

tokenizer.model:   0%|          | 0.00/587k [00:00<?, ?B/s]

...

model.safetensors.index.json: 0.00B [00:00, ?B/s]

Fetching 3 files:   0%|          | 0/3 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/291 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

Model loaded successfully ✅


## Server

In [4]:
def run_server():
    print("Starting server...")
    app.run(host="0.0.0.0", port=8000, threaded=True)
    
server_thread = threading.Thread(target=run_server, daemon=True)
server_thread.start()

time.sleep(5)
print("Thread alive?", server_thread.is_alive())

ngrok.kill()
ngrok.set_auth_token("3Jv6ZQVEetePr3NPOweex5XfGul_4dfDHH2bKk7kc7LzB5CoQ")
public_url = ngrok.connect(8000)
print("Public URL:", public_url)

Starting server...
 * Serving Flask app '__main__'
 * Debug mode: off


 * Running on all addresses (0.0.0.0)
 * Running on http://127.0.0.1:8000
 * Running on http://172.19.2.2:8000
Press CTRL+C to quit


Thread alive? True
Public URL: NgrokTunnel: "https://stinging-gruffly-progress.ngrok-free.dev" -> "http://localhost:8000"


In [5]:
response = requests.post(
    "http://127.0.0.1:8000/generate",
    json={"prompt": "Say only the word: OK", "max_new_tokens": 20},
    timeout=60
)

print("Status:", response.status_code)
print("Raw text:", response.text)

127.0.0.1 - - [28/Sep/2026 15:36:41] "POST /generate HTTP/1.1" 200 -


Status: 200
Raw text: {"response":"OK"}



## Test 

In [6]:
print(server_thread.is_alive())

True


In [7]:
s = socket.socket(socket.AF_INET, socket.SOCK_STREAM)
result = s.connect_ex(("127.0.0.1", 8000))
print("Port open:", result == 0)
s.close()

Port open: True


## Thank You